In [27]:
from qdrant_client import QdrantClient, models
from openai import OpenAI
import numpy as np
import uuid
from transformers import AutoTokenizer
from docling_core.types.doc import DoclingDocument
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from docling_core.transforms.chunker.hybrid_chunker import HybridChunker
from pathlib import Path

In [28]:
EXPORT = Path('/home/pinch/projects/vLLM_demo/rag/demo/export_folder')

In [29]:
embedder_url = 'http://127.0.0.1:8001/v1'
qdrant_url = 'http://127.0.0.1:6333'
EMBEDDING_MODEL_NAME = 'bge-m3'


embedder = OpenAI(
  base_url=embedder_url,
  api_key='nokey'
)

qclient = QdrantClient(
  url=qdrant_url,
  api_key="1234"
)

/tmp/ipykernel_6891/3291360312.py:11: UserWarning: Api key is used with an insecure connection.
  qclient = QdrantClient(


In [30]:
tokenizer = HuggingFaceTokenizer(tokenizer=AutoTokenizer.from_pretrained('rag_models/embedder/bge-m3'), max_tokens=128)
chunker = HybridChunker(tokenizer=tokenizer)

In [40]:
def get_chunks(document_path, chunker):
    document = DoclingDocument.load_from_json(document_path)
    chunks = chunker.chunk(document)
    data = []
    for i, chunk in enumerate(chunks):
        data.append(
            {
                'chunk_id': i+1,
                'document': document_path.name,
                'text': chunker.contextualize(chunk)
            }
        )
    return data

In [32]:
def get_dense_vector(text, embedder):
  r = embedder.embeddings.create(model=EMBEDDING_MODEL_NAME, input=text)
  return np.array(r.data[0].embedding)


In [33]:
sparse_vector_options = {
  'language': 'french',
  'ascii_folding': True,
  'avg_len': 250
}

def get_sparse_vector(text):
  return models.Document(text=text, model='qdrant/bm25', options=sparse_vector_options)

In [34]:
def get_point_struct(chunk, embedder):
  text = chunk.get('text')
  chunk_id = chunk.get('chunk_id')
  file_path = chunk.get('document')
  
  payload = {
  'file_extension': 'pdf',
  'chunk_id': chunk_id,
  'file_path': file_path,
  'extraction_date': '2026-01-01',
  'text': text
  }

  vector = get_dense_vector(text, embedder)
  sparse_vector = get_sparse_vector(text)
  id = str(uuid.uuid5(uuid.NAMESPACE_URL, f'{chunk_id} - {file_path}'))

  point = models.PointStruct(
    id = id,
    vector = 
      {
        'dense': vector,
        'sparse': sparse_vector
      },
      payload = payload
  )
  
  return point
  

In [35]:
def get_points(chunks, embedder):
    return [get_point_struct(chunk, embedder) for chunk in chunks]    


In [36]:
collection_name = 'demo_collection'

if not qclient.collection_exists(collection_name):
  qclient.create_collection(
    collection_name,
    vectors_config={
      'dense': models.VectorParams(size=1024, distance=models.Distance.COSINE),
    },
    sparse_vectors_config={
      'sparse': models.SparseVectorParams(modifier=models.Modifier.IDF)
    } )


In [37]:
def get_all_chunks(docs):
  all_chunks = []
  for doc in docs:
    chunks = get_chunks(doc, chunker) 
    all_chunks.extend(chunks)
  return all_chunks

In [41]:
docs = sorted(EXPORT.glob('*.json'))
all_chunks = get_all_chunks(docs)
print('all_chunks: ', all_chunks)
points = get_points(all_chunks, embedder)
qclient.create_payload_index(collection_name, field_name='file_path', field_schema=models.PayloadSchemaType.KEYWORD)
qclient.upsert(collection_name, points=points)

all_chunks:  [{'chunk_id': 1, 'document': 'grille_salariale_2026.pdf.json', 'text': "Grille des rémunérations et avantages\nApplicable au 1er janvier 2026\nCette grille fixe le salaire mensuel brut minimum de chaque fonction en fonction de l'ancienneté dans la classe. Elle est revue chaque année en janvier après concertation avec la délégation du personnel. Les montants s'entendent pour un temps plein."}, {'chunk_id': 2, 'document': 'grille_salariale_2026.pdf.json', 'text': '1. Barème des salaires de base\n\nOuvriers, Catégorie Classe = A - Opérateur. Ouvriers, Salaire mensuel brut de base.Échelon 1 (0-2 ans) = 2 280 €. Ouvriers, Salaire mensuel brut de base.Échelon 2 (3-5 ans) = 2 390 €. Ouvriers, Salaire mensuel brut de base.Échelon 3 (6-10 ans) = 2 520 €. Ouvriers, Salaire mensuel brut de base.Échelon 4 (plus de 10 ans) = 2 660 €. Ouvriers, Catégorie Classe = B'}, {'chunk_id': 3, 'document': 'grille_salariale_2026.pdf.json', 'text': '1. Barème des salaires de base\n\n - Opérateur qu

UpdateResult(operation_id=3, status=<UpdateStatus.COMPLETED: 'completed'>)

In [ ]:
docs = EXPORT.glob('*.json')
all_chunks = []
for doc in docs:
    chunks = get_chunks(doc, chunker)
    all_chunks.extend(chunks)
all_chunks

[{'chunk_id': 1,
  'document': 'politique_teletravail_2025.docx.json',
  'text': 'Politique de télétravail\n*Version 2.0 — en vigueur au 1er octobre 2025 — remplace la version 1.0 du 1er mars 2023*'},
 {'chunk_id': 2,
  'document': 'politique_teletravail_2025.docx.json',
  'text': 'Politique de télétravail\n1. Objet\nCette politique définit les conditions dans lesquelles le personnel de Veltis peut travailler depuis son domicile.'},
 {'chunk_id': 3,
  'document': 'politique_teletravail_2025.docx.json',
  'text': "Politique de télétravail\n2. Éligibilité\nLe télétravail est ouvert aux fonctions dont les tâches peuvent être réalisées à distance, après la période d'essai. Les postes de production et de logistique ne sont pas éligibles."},
 {'chunk_id': 4,
  'document': 'politique_teletravail_2025.docx.json',
  'text': "Politique de télétravail\n3. Organisation\nLe télétravail est limité à 3 jours par semaine. La présence sur site est obligatoire le mardi, journée consacrée aux réunions d'

In [ ]:


r = embedder.embeddings.create(model="bge-m3", input=text)
vector = np.array(r.data[0].embedding)
vector

array([-0.05836442, -0.01487188, -0.01153062, ...,  0.00048697,
       -0.0244277 ,  0.02023475], shape=(1024,))

In [33]:


file_path = 'export_data/exemple.pdf'
chunk_id = 1

id = str(uuid.uuid5(uuid.NAMESPACE_URL, f'{chunk_id} - {file_path}'))
id

'65113525-6fa1-52f1-9b4c-fc7045515139'

In [34]:
sparse_vector_options = {
  'language': 'french',
  'ascii_folding': True,
  'avg_len': 50
}

sparse_vector = models.Document(text=text, model='qdrant/bm25', options=sparse_vector_options)
sparse_vector

Document(text="Les dividendes pour l'année 2026 sront de 10M", model='qdrant/bm25', options={'language': 'french', 'ascii_folding': True, 'avg_len': 50})

In [ ]:

payload = {
  'file_extension': 'pdf',
  'chunk_id': chunk.id,
  'file_path': file_path,
  'extraction_date': '2026-01-01',
  'text': text
}

point = models.PointStruct(
  id = id,
  vector = {'dense': vector, 'sparse': sparse_vector},
  payload=payload
)

In [ ]:
collection_name = 'demo_collection'

if not qclient.collection_exists(collection_name):
  qclient.create_collection(
    collection_name,
    vectors_config={
      'dense': models.VectorParams(size=1024, distance=models.Distance.COSINE),
    },
    sparse_vectors_config={
      'sparse': models.SparseVectorParams(modifier=models.Modifier.IDF)
    } )

  qclient.update_collection(collection_name, 
                            )

In [31]:
qclient.create_payload_index(collection_name, field_name='file_path', field_schema=models.PayloadSchemaType.KEYWORD)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

In [36]:
qclient.upsert(collection_name, points=[point,])

UpdateResult(operation_id=3, status=<UpdateStatus.COMPLETED: 'completed'>)